#**CHAPTER 4.A COMMITTEE**
---

##REFERENCE

https://chatgpt.com/share/6998c74e-e778-8012-a17b-3196ad0030c8

##0.CONTEXT

**INTRODUCTION (Board-Facing, Governance-First) — Notebook 4: Committee Reasoning Under Regime Uncertainty**

You are looking at a controlled experiment: a reasoning pipeline that simulates how an investment committee should evaluate a proposed allocation when the environment is uncertain and when we cannot rely on “hand-wavy” intuition or undocumented analyst judgment. The purpose of this notebook is not to predict markets, pick winners, or automate fiduciary decisions. The purpose is to demonstrate—clearly, auditablely, and repeatably—how an AI component can be used inside an institutional process while still keeping humans accountable and keeping the system reviewable by a Board, Risk Committee, Compliance, and auditors.

In a traditional setting, an analyst might write a memo, circulate it, hold a meeting, and capture a decision in minutes. The risk is not the meeting itself; the risk is the invisibility of the reasoning. If the analysis is not traceable, if assumptions are not labeled, if dissent is not preserved, and if the decision rule is not explicit, then the organization has a governance gap. Later, when outcomes disappoint—or when regulators ask “why did you do this?”—we cannot reconstruct what the organization believed, what it knew, what it guessed, what it debated, and what conditions would have changed the decision.

This notebook addresses that governance gap by engineering the committee process as a pipeline with explicit boundaries, explicit role outputs, explicit voting and policy rules, and explicit audit artifacts. Think of it as converting a “meeting + memo” into a “controlled process + evidence bundle.” The evidence bundle is created on every run and includes: a run manifest (what was executed), a prompt log (what we asked the model, with redaction and hashing), a reasoning trace (what each role concluded and how the supervisor synthesized), a risk log (what controls triggered), and a final board-facing report (what we recommend and what must be verified). This is the minimum standard for using an AI component in a decision workflow without treating it as magic.

**What problem are we solving?**
We are solving the problem of institutional reasoning under uncertainty. The use case is intentionally realistic: allocating to a thematic equity basket when regimes can shift (risk-on to risk-off, inflation shocks, growth slowdowns). In such settings, reasonable professionals can disagree. The key governance requirement is not to eliminate disagreement; it is to surface it, preserve it, and convert it into conditions and tests. That is exactly what the committee structure does: it forces multiple perspectives (Portfolio, Risk, Compliance/Suitability, Macro) to state their stance, arguments, objections, and required conditions.

**Why “committee reasoning” is the right architecture for Boards**
Boards do not want a single model output. Boards want: (1) an intelligible record of competing viewpoints, (2) an explicit decision rule, (3) a clear list of what is known vs assumed, (4) a list of open items, and (5) a defensible escalation path. Committee reasoning encodes these requirements directly:
- It produces one memo per role (four independent lenses).
- It forces a vote tally.
- It preserves dissent when stances differ.
- It enforces a policy veto (Compliance/Suitability can force escalation).
- It produces an auditable “committee_record” object that can be reviewed later.

**What inputs are used (and what is deliberately out of scope)**
The notebook uses deterministic synthetic inputs: client constraints (risk tolerance band, max drawdown tolerance, liquidity needs), a synthetic regime indicator (with confidence), and synthetic basket characteristics (volatility proxy, correlation proxy, concentration proxy, liquidity profile). This is deliberate. We are not trying to “prove performance.” We are proving process. The process must remain valid even when market data is missing, ambiguous, or later revised. The notebook therefore forbids the model from inventing tickers, returns, forward-looking claims, or external sources. If a fact is not provided, it must be flagged as unknown and added as an open item.

**How the reasoning pipeline works, in plain terms**
1) We generate a bounded case packet (facts).  
2) We ask each role to produce a structured memo (stance + arguments + objections + conditions + risk flags).  
3) We validate each memo against a strict schema. If invalid, we log a risk and insert a placeholder that forces human review.  
4) We ask a supervisor to synthesize the committee record: vote tally, dissent log, recommendation, escalation requirement.  
5) We validate the committee record against a strict schema. If it fails, we use a deterministic fallback synthesis and escalate.  
6) We apply governance gates:
   - Role completeness: all four roles must be present.
   - Dissent preservation: if there is disagreement, dissent must be recorded.
   - Policy enforcement: compliance veto is binding.
7) We write a board-facing final report that separates facts, assumptions, open items, analysis, and draft output, and labels verification as “Not verified.”

**What “results” mean in this notebook**
The outputs are not “the correct investment decision.” The outputs are:
- A complete, reviewable record of the committee’s reasoning structure.
- A decision outcome that is explicitly linked to the policy rules and gates.
- A list of uncertainties and required verification items.
- A reproducible artifact bundle that can be audited.

In other words, the result is institutional defensibility. It demonstrates that the organization can use AI as a bounded assistant while maintaining: (a) explicit controls, (b) human accountability, and (c) a documented chain of reasoning and dissent.

**Why this is relevant now**
Three pressures are converging: speed, complexity, and scrutiny. Markets move faster, products are more complex, and regulators / clients demand higher documentation. AI can increase throughput, but only if we match increased capability with increased controls. This notebook is a practical demonstration of that principle: capability up implies risk up implies controls up. We are not asking the Board to trust a model; we are asking the Board to approve a controlled process in which model outputs are treated as inputs to governance, not as decisions.

**How to interpret the artifacts**
- run_manifest.json: “What did we run, with which configuration?”  
- prompts_log.jsonl: “What did we ask (redacted), and what is the hash of the exact prompt?”  
- reasoning_trace.json: “What did each role say, what was the tally, what dissent exists, and which gates passed?”  
- risk_log.json: “Which control failures or policy triggers occurred?”  
- final_report.json: “Board-facing summary with explicit separation of facts/assumptions/open items and a governed decision.”  
- deliverables.zip: “Packaged evidence bundle for distribution and archiving.”

This is the foundation we need if we want AI in finance to be usable in Board environments: not a demo, but a controlled system that creates evidence.

**Bottom line**
This notebook shows how to operationalize a committee-style reasoning process with AI components without sacrificing governance. It provides a repeatable template for multi-role analysis, dissent preservation, policy enforcement, and audit-grade artifact generation. It is a process demonstration first, and only secondarily a content generator. That ordering is intentional, and it is what makes the work relevant for a Board of Directors.

##1.LIBRARIES AND ENVIRONMENT

**CELL 1/10 — Install, Imports, Determinism, and Directory Setup (Pedagogical Explanation)**

This first cell establishes the notebook’s “operating environment” and the minimum governance posture before we do any reasoning. It does four essential things: (1) installs dependencies, (2) imports the libraries we will use, (3) applies best-effort determinism controls, and (4) creates a predictable artifact directory structure.

First, we pin and install only the packages we need: the Anthropic client for calling Claude and jsonschema for enforcing output validity. In a governance-first workflow, fewer dependencies is a feature, not a limitation: every extra library adds operational uncertainty and expands the audit surface. Pinning versions reduces “it worked yesterday but not today” issues, which is a common failure mode in analytical pipelines.

Second, we import standard libraries for time, hashing, random generation, file handling, and typing. Typing matters because this notebook is building “control-grade” objects: role memos, committee records, risk logs, and final reports. Even in Python, basic structure and type discipline reduces ambiguity and helps reviewers trust what they are reading.

Third, we apply determinism controls. We set PYTHONHASHSEED and random.seed. This does not make the entire universe deterministic (LLM calls can introduce variation), but it does ensure that synthetic data generation and local operations are repeatable. For Boards and audit functions, repeatability is a core expectation: if we can rerun the notebook and get the same case packet, we can isolate changes to the model output or configuration instead of arguing about whether the data itself moved.

Fourth, we create two directories: artifacts/ and deliverables/. This is a governance requirement. A notebook is not “done” when it prints text; it is done when it produces stable files that can be archived and reviewed. The rest of the notebook will write a manifest, logs, traces, and reports into these folders.

Finally, the cell generates a run_id derived from a timestamp and seed context. The run_id is the unique key that ties together every file written during the run. That single design choice—consistent run identifiers—makes this pipeline reviewable, searchable, and suitable for later incident reconstruction.

In [1]:
# CELL 1/10 — Install + imports + deterministic settings + directory setup
!pip -q install anthropic==0.45.0 jsonschema==4.23.0

import os, re, json, time, hashlib, random, zipfile, textwrap, pathlib
import datetime
from typing import Any, Dict, List, Optional, Tuple, TypedDict, Literal

from jsonschema import Draft202012Validator, ValidationError
from google.colab import userdata

ARTIFACTS_DIR = pathlib.Path("artifacts")
DELIVERABLES_DIR = pathlib.Path("deliverables")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)

# Determinism controls (best-effort within notebook constraints)
os.environ["PYTHONHASHSEED"] = "0"
random.seed(1337)

def utc_now_iso() -> str:
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

RUN_ID = hashlib.sha256(f"{utc_now_iso()}|committee|seed=1337".encode("utf-8")).hexdigest()[:16]
print("run_id:", RUN_ID)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.3/222.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 10.8 MB/s eta 0:00:00
run_id: 7492e5e00d94ca0b


##2.CONFIGURATION AND SCHEMAS

###2.1.OVERVIEW

**CELL 2/10 — Configuration, Schemas, and Governance Helpers (Pedagogical Explanation)**

This cell is the governance “spine” of the notebook. It defines what valid outputs look like, how we record evidence, how we redact sensitive content, and how we detect and document failures. In board terms: this cell defines the rules of the game before we let an AI component speak.

The first block defines helper functions for writing JSON and JSONL. JSON is used for structured artifacts like the run manifest and final report; JSONL is used for append-only logs like the prompts log. Append-only logging is important because it behaves like a ledger: it is harder to “rewrite history” accidentally.

Next, the cell defines hashing utilities. We hash prompts and JSON objects so we can prove exactly what was asked and what configuration was used without storing sensitive or bulky content. Hashes are the bridge between transparency and privacy: you can share a log of hashes with auditors and still protect details.

Redaction is defined next. The notebook explicitly avoids writing secrets (API keys) and redacts email/phone-like patterns. Even if our synthetic cases do not include personal data, the habit is important: in real deployments, a boundary failure could cause PII to appear in prompts. This cell ensures we do not casually log it.

The most important section is schema definition. We define a strict schema for a role memo: every role must output role, stance, arguments, objections, required conditions, and risk flags. This prevents “pretty paragraphs” that are impossible to compare across roles. It forces structure and keeps the committee process crisp.

We then define a schema for the committee record, including vote tally, dissent log, synthesis recommendation, and escalation required. This is the mechanism that makes dissent “non-optional.” If stances differ, dissent must exist as a structured field that can be audited.

Finally, we define the final_report schema. This enforces separation of facts_provided, assumptions_introduced, open items/questions to verify, analysis, and draft output—and it forces verification_status = “Not verified.” That last requirement matters: it prevents the notebook from silently sounding authoritative. The report must communicate uncertainty clearly.

In summary, Cell 2 turns qualitative reasoning into a set of enforceable contracts. If outputs do not match the contracts, the notebook will log risks and escalate instead of guessing.

###2.2.CODE AND IMPLEMENTATION

In [2]:
# CELL 2/10 — Config + schemas + helpers (hashing, redaction, JSON writing, validation)
MODEL_NAME = "claude-haiku-4-5-20251001"

def write_json(path: str, obj: Any) -> None:
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, sort_keys=True)

def append_jsonl(path: str, obj: Any) -> None:
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, sort_keys=True) + "\n")

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def sha256_json(obj: Any) -> str:
    return sha256_text(json.dumps(obj, ensure_ascii=False, sort_keys=True))

_RE_EMAIL = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
_RE_PHONE = re.compile(r"\b(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{2,4}\)?[-.\s]?)?\d{3,4}[-.\s]?\d{4}\b")
_RE_KEYLIKE = re.compile(r"\b(sk-[A-Za-z0-9]{8,}|ANTHROPIC_[A-Za-z0-9_]{8,}|[A-Za-z0-9_-]{24,})\b")

def redact_text(s: str) -> str:
    s2 = _RE_EMAIL.sub("[REDACTED_EMAIL]", s)
    s2 = _RE_PHONE.sub("[REDACTED_PHONE]", s2)
    s2 = _RE_KEYLIKE.sub("[REDACTED_SECRET]", s2)
    return s2

def safe_extract_json(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    candidate = text[start:end+1]
    try:
        return json.loads(candidate)
    except Exception:
        return None

def validate_or_none(schema: Dict[str, Any], obj: Any) -> Tuple[bool, Optional[str]]:
    try:
        Draft202012Validator(schema).validate(obj)
        return True, None
    except ValidationError as e:
        return False, str(e)

def make_risk(
    severity: Literal["LOW","MEDIUM","HIGH"],
    category: str,
    description: str,
    control: str,
    status: Literal["OPEN","MITIGATED","CLOSED"]="OPEN",
    risk_id: Optional[str]=None
) -> Dict[str, Any]:
    rid = risk_id or hashlib.sha256(f"{RUN_ID}|{utc_now_iso()}|{category}|{description}".encode("utf-8")).hexdigest()[:12]
    return {
        "risk_id": rid,
        "timestamp_utc": utc_now_iso(),
        "severity": severity,
        "category": category,
        "description": description,
        "control": control,
        "status": status
    }

ROLE_MEMO_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "required": ["role","stance","key_arguments","objections","required_conditions","risk_flags"],
    "properties": {
        "role": {"type": "string"},
        "stance": {"type": "string", "enum": ["APPROVE","REJECT","REVIEW"]},
        "key_arguments": {"type": "array", "items": {"type": "string"}, "minItems": 2, "maxItems": 8},
        "objections": {"type": "array", "items": {"type": "string"}, "minItems": 1, "maxItems": 8},
        "required_conditions": {"type": "array", "items": {"type": "string"}, "minItems": 1, "maxItems": 8},
        "risk_flags": {"type": "array", "items": {"type": "string"}, "minItems": 1, "maxItems": 10}
    }
}

COMMITTEE_RECORD_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "required": ["committee_record"],
    "properties": {
        "committee_record": {
            "type": "object",
            "additionalProperties": False,
            "required": ["role_memos","vote_tally","dissent_log","synthesis_recommendation","escalation_required"],
            "properties": {
                "role_memos": {"type": "array", "items": ROLE_MEMO_SCHEMA, "minItems": 4, "maxItems": 4},
                "vote_tally": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["APPROVE","REJECT","REVIEW"],
                    "properties": {
                        "APPROVE": {"type": "integer", "minimum": 0, "maximum": 4},
                        "REJECT": {"type": "integer", "minimum": 0, "maximum": 4},
                        "REVIEW": {"type": "integer", "minimum": 0, "maximum": 4}
                    }
                },
                "dissent_log": {"type": "array", "items": {"type": "string"}, "minItems": 0, "maxItems": 30},
                "synthesis_recommendation": {"type": "string"},
                "escalation_required": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["required","why"],
                    "properties": {
                        "required": {"type": "boolean"},
                        "why": {"type": "string"}
                    }
                }
            }
        }
    }
}

FINAL_REPORT_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "run_id","timestamp_utc","executive_summary","facts_provided","assumptions_introduced",
        "open_items","questions_to_verify","analysis","draft_output",
        "committee_summary","final_decision","confidence","verification_status"
    ],
    "properties": {
        "run_id": {"type": "string"},
        "timestamp_utc": {"type": "string"},
        "executive_summary": {"type": "string"},
        "facts_provided": {"type": "object"},
        "assumptions_introduced": {"type": "array", "items": {"type": "string"}},
        "open_items": {"type": "array", "items": {"type": "string"}},
        "questions_to_verify": {"type": "array", "items": {"type": "string"}},
        "analysis": {"type": "string"},
        "draft_output": {"type": "string"},
        "committee_summary": {
            "type": "object",
            "additionalProperties": False,
            "required": ["per_role_summaries","vote_tally","dissent","policy_notes"],
            "properties": {
                "per_role_summaries": {"type": "array", "items": {"type": "object"}, "minItems": 4, "maxItems": 4},
                "vote_tally": {"type": "object"},
                "dissent": {"type": "array", "items": {"type": "string"}},
                "policy_notes": {"type": "array", "items": {"type": "string"}}
            }
        },
        "final_decision": {"type": "string", "enum": ["APPROVE","REJECT","HUMAN_REVIEW"]},
        "confidence": {"type": "string", "enum": ["low","medium","high"]},
        "verification_status": {"type": "string", "enum": ["Not verified"]}
    }
}

CONFIG: Dict[str, Any] = {
    "model": MODEL_NAME,
    "max_tokens_role": 650,
    "max_tokens_synth": 750,
    "temperature": 0.0,
    "policy": {
        "compliance_veto": True,   # if Compliance says REJECT => final must be HUMAN_REVIEW or REJECT
        "compliance_veto_outcome": "HUMAN_REVIEW"
    },
    "limits": {
        "max_prompt_chars_logged": 2000
    }
}

print("config_hash:", sha256_json(CONFIG)[:16])

config_hash: f481e43b1f1f9f45


##3.SYNTHETIC FINANCE CASE GENERATOR

###3.1.OVERVIEW

**CELL 3/10 — Synthetic Case Generation and Input Boundary (Pedagogical Explanation)**

This cell creates the “case packet” that the committee will evaluate. It is synthetic by design: the goal is to validate the reasoning pipeline, not to claim real market insight. Using synthetic inputs removes confusion about whether the notebook is “backtesting” or “predicting” anything. It is not. It is demonstrating process.

The case includes: (1) client constraints, (2) a regime indicator, and (3) basket characteristics. Each element is expressed in plain, bounded terms. For example, the client has a risk tolerance band and a stated drawdown tolerance. The regime indicator is explicitly synthetic and includes a confidence label. The basket has synthetic proxies for volatility, correlation, concentration (HHI proxy), and liquidity profile.

The most important governance component is the input boundary. The boundary explicitly lists which keys count as “allowed facts” and what categories are prohibited (real tickers, real returns, “external sources,” forward-looking guarantees). This boundary is the contract that the model must respect. Without it, a model can easily drift into invented specifics (“this basket historically returns X%”) that sound plausible but are not provided.

The cell also writes the first required artifact: run_manifest.json. The manifest records the run_id, timestamp, notebook identity, reasoning pattern (COMMITTEE), model name, config hash, and determinism settings. For governance, this is crucial. When a Board asks “what produced this report?”, we point to the manifest: it specifies which version of the pipeline was executed and with what configuration.

Because the case is deterministic (seeded), the same case can be regenerated and re-reviewed. That matters for committee training and operational readiness: you can run the notebook multiple times and validate that controls and gates behave consistently.

In short, Cell 3 defines the “facts provided” and writes them into an audit context. From this point onward, the rest of the notebook is not allowed to add facts—only to interpret, condition, and escalate based on missing information.

###3.2.CODE AND IMPLEMENTATION

In [3]:
# CELL 3/10 — Synthetic finance case generator (deterministic) + input boundary definition
def make_synthetic_case(seed: int = 1337) -> Dict[str, Any]:
    rng = random.Random(seed)
    themes = ["AI Infrastructure", "Energy Transition", "Cybersecurity", "Semiconductors", "Healthcare Innovation"]
    theme = themes[rng.randrange(len(themes))]
    regime = rng.choice(["Risk-on", "Risk-off", "Inflation-up", "Growth-slowdown"])

    case = {
        "case_id": f"SYN-COMMITTEE-{seed}",
        "timestamp_utc": utc_now_iso(),
        "investment_decision": "Allocate to a thematic equity basket under regime uncertainty",
        "client_constraints": {
            "risk_tolerance_band": rng.choice(["Conservative", "Moderate", "Growth"]),
            "max_drawdown_tolerance_pct_stated": rng.choice([8, 10, 12, 15]),
            "liquidity_needs": rng.choice(["Monthly liquidity needed", "Quarterly liquidity acceptable", "No near-term liquidity needs"]),
            "time_horizon_years": rng.choice([3, 5, 7])
        },
        "regime_indicator": {
            "indicator_name": "Synthetic Macro Regime Signal",
            "current_regime": regime,
            "confidence": rng.choice(["low", "medium", "high"]),
            "notes": "Synthetic indicator; illustrative only. Not market data."
        },
        "basket_characteristics": {
            "theme": theme,
            "num_names": rng.choice([12, 15, 20]),
            "volatility_annualized_pct": rng.choice([18, 22, 26, 30]),
            "correlation_proxy": rng.choice(["low", "medium", "high"]),
            "concentration_hhi_proxy": rng.choice(["low", "medium", "high"]),
            "liquidity_profile": rng.choice(["large-cap liquid", "mixed liquidity", "some small/mid liquidity risk"]),
            "notes": "All characteristics are synthetic proxies; do not treat as real estimates."
        },
        "input_boundary": {
            "allowed_facts_keys": [
                "investment_decision","client_constraints","regime_indicator","basket_characteristics"
            ],
            "prohibited": [
                "real tickers", "real market returns", "real vol estimates", "forward-looking guarantees", "external sources not provided"
            ]
        }
    }
    return case

CASE = make_synthetic_case(1337)
write_json("artifacts/run_manifest.json", {
    "run_id": RUN_ID,
    "timestamp_utc": utc_now_iso(),
    "notebook": "Notebook 4 — Committee Reasoning",
    "pattern": "COMMITTEE",
    "model": MODEL_NAME,
    "config_hash": sha256_json(CONFIG),
    "determinism": {
        "seed": 1337,
        "PYTHONHASHSEED": os.environ.get("PYTHONHASHSEED",""),
        "note": "Notebook-level best-effort determinism."
    },
    "inputs": {
        "case_id": CASE["case_id"],
        "input_boundary": CASE["input_boundary"]
    }
})
print(json.dumps(CASE, indent=2, ensure_ascii=False)[:1200])

{
  "case_id": "SYN-COMMITTEE-1337",
  "timestamp_utc": "2026-02-20T20:22:01.974042+00:00",
  "investment_decision": "Allocate to a thematic equity basket under regime uncertainty",
  "client_constraints": {
    "risk_tolerance_band": "Growth",
    "max_drawdown_tolerance_pct_stated": 10,
    "liquidity_needs": "Quarterly liquidity acceptable",
    "time_horizon_years": 5
  },
  "regime_indicator": {
    "indicator_name": "Synthetic Macro Regime Signal",
    "current_regime": "Inflation-up",
    "confidence": "high",
    "notes": "Synthetic indicator; illustrative only. Not market data."
  },
  "basket_characteristics": {
    "theme": "Healthcare Innovation",
    "num_names": 15,
    "volatility_annualized_pct": 26,
    "correlation_proxy": "medium",
    "concentration_hhi_proxy": "high",
    "liquidity_profile": "large-cap liquid",
    "notes": "All characteristics are synthetic proxies; do not treat as real estimates."
  },
  "input_boundary": {
    "allowed_facts_keys": [
      "inv

##4.LLM WRAPPER

###4.1.OVERVIEW

**CELL 4/10 — LLM Client Wrapper and Prompt Logging (Pedagogical Explanation)**

This cell is where the notebook connects to Claude, but it does so in a controlled way. The key principle is: we never call the model directly; we call it through a wrapper that enforces logging, redaction, hashing, and error handling.

First, we read the API key from Colab Secrets (ANTHROPIC_API_KEY). This prevents accidental exposure of secrets in notebook text or logs. Governance begins with credential hygiene: the pipeline must be safe even if people share the notebook.

Next, we define prompt logging to artifacts/prompts_log.jsonl. Every model request is recorded with a timestamp, run_id, role, model name, temperature, max tokens, and a hash of the exact prompt. We also store a redacted version of the prompt with strict length limits. This is not bureaucratic; it is how we reconstruct what the model was asked and ensure the process is reviewable.

We then define call_claude_json, which wraps the model call and attempts to parse JSON from the response. This wrapper is key to governance. Our pipeline requires structured outputs that can be validated against schemas. If the model returns non-JSON narrative, we treat that as a failure condition, not as content we “interpret.”

If the model call fails (network errors, API issues, unexpected response), the wrapper logs a HIGH severity risk. Crucially, it does not silently proceed as if nothing happened. This protects decision workflows from “partial failure” states where a missing memo might be mistakenly treated as a neutral stance.

This cell also centralizes risk logging integration. Rather than scattering ad-hoc print statements, we treat risks as structured objects: risk_id, timestamp, severity, category, description, control, status. That makes risk reporting consistent across runs and makes it possible to build dashboards later.

In board terms, Cell 4 is the line between “AI as a black box” and “AI as a bounded component inside a controlled system.” It ensures that every model interaction leaves an audit trail, and every failure becomes a governance signal rather than an invisible gap.

###4.2.CODE AND IMPLEMENTATION

In [4]:
# CELL 4/10 — LLM client wrapper (Anthropic) + prompt logging (redacted + hashes)
from anthropic import Anthropic

API_KEY = userdata.get("ANTHROPIC_API_KEY")
if not API_KEY or not isinstance(API_KEY, str):
    raise RuntimeError("Missing ANTHROPIC_API_KEY in Colab Secrets (google.colab.userdata).")

client = Anthropic(api_key=API_KEY)

PROMPTS_LOG_PATH = "artifacts/prompts_log.jsonl"
RISK_LOG_PATH = "artifacts/risk_log.json"
TRACE_PATH = "artifacts/reasoning_trace.json"
FINAL_REPORT_PATH = "artifacts/final_report.json"

_risks: List[Dict[str, Any]] = []
def log_risk(r: Dict[str, Any]) -> None:
    _risks.append(r)

def log_prompt(kind: str, role: str, prompt: str, meta: Dict[str, Any]) -> None:
    red = redact_text(prompt)
    red = red[:CONFIG["limits"]["max_prompt_chars_logged"]]
    entry = {
        "run_id": RUN_ID,
        "timestamp_utc": utc_now_iso(),
        "kind": kind,                   # "role_memo" or "synthesis"
        "role": role,
        "model": MODEL_NAME,
        "temperature": CONFIG["temperature"],
        "max_tokens": meta.get("max_tokens"),
        "prompt_sha256": sha256_text(prompt),
        "redacted_prompt": red
    }
    append_jsonl(PROMPTS_LOG_PATH, entry)

def call_claude_json(system: str, user: str, kind: str, role: str, max_tokens: int) -> Tuple[Optional[Dict[str, Any]], str]:
    prompt_for_log = f"SYSTEM:\n{system}\n\nUSER:\n{user}"
    log_prompt(kind=kind, role=role, prompt=prompt_for_log, meta={"max_tokens": max_tokens})
    try:
        msg = client.messages.create(
            model=MODEL_NAME,
            max_tokens=max_tokens,
            temperature=CONFIG["temperature"],
            system=system,
            messages=[{"role":"user","content": user}],
        )
        text = ""
        try:
            # anthropic SDK returns content list of blocks
            if hasattr(msg, "content") and msg.content:
                text = "".join([getattr(b, "text", "") for b in msg.content])
            else:
                text = str(msg)
        except Exception:
            text = str(msg)
        obj = safe_extract_json(text)
        return obj, text
    except Exception as e:
        log_risk(make_risk(
            severity="HIGH",
            category="LLM_CALL_FAILURE",
            description=f"Anthropic call failed for {role}: {type(e).__name__}: {e}",
            control="Retry not implemented; escalate to HUMAN_REVIEW and continue with placeholders."
        ))
        return None, ""

##5.ORCHESTATION

###5.1.OVERVIEW

**CELL 5/10 — Committee Orchestration: Four Roles + Supervisor Synthesis (Pedagogical Explanation)**

This is the heart of the notebook: it operationalizes the committee pattern. The key idea is separation of perspectives. A single memo, even from a strong analyst, is vulnerable to blind spots. Committee reasoning forces independent lenses to state their stance and conditions.

We define exactly four roles: Portfolio Manager, Risk Officer, Compliance/Suitability, and Macro Strategist. This is not arbitrary: each role corresponds to a real institutional function. Portfolio focuses on return logic and fit; Risk focuses on drawdown and concentration; Compliance focuses on suitability and constraints; Macro focuses on regime interpretation and scenario framing.

For each role, we build a user prompt that includes: the bounded facts (only allowed keys), strict instructions not to invent facts, and a schema hint. The role must respond in JSON with stance, key arguments, objections, required conditions, and risk flags. This structure is what makes the outputs comparable and reviewable.

We then run one LLM call per role. This mirrors a real committee: each participant speaks for themselves. The notebook validates each role memo against the ROLE_MEMO_SCHEMA. If validation fails or the memo is missing, the notebook logs a HIGH risk and inserts a placeholder memo that forces REVIEW and escalates. This is a critical governance pattern: when the system cannot guarantee structured integrity, it does not “guess” the missing perspective.

After collecting role memos, the notebook calls a supervisor synthesis. The supervisor’s job is not to overwrite roles; it is to assemble the committee record: vote tally, dissent log, synthesis recommendation, escalation requirement. The supervisor is required to preserve dissent when stances differ, and to apply compliance policy rules (though final enforcement occurs in later gates).

The result of Cell 5 is a set of role memos plus an initial committee record candidate. Think of this cell as the “meeting simulation.” But unlike real meetings, it forces structure and creates machine-checkable artifacts.

In board terms: this is how we convert multi-voice deliberation into a standardized record that can be audited and discussed without losing nuance or dissent.

###5.2.CODE AND IMPLEMENTATION

In [5]:
# CELL 5/10 — Committee orchestration (4 roles + synthesis)  [MUST IMPLEMENT]
Roles = [
    "Portfolio Manager",
    "Risk Officer",
    "Compliance/Suitability",
    "Macro Strategist"
]

ROLE_SYSTEM = (
    "You are a disciplined finance committee member. "
    "You must follow the input boundary: do not invent facts, tickers, market data, or external sources. "
    "If information is missing, list it as open items / required conditions. "
    "Return ONLY valid JSON matching the required schema."
)

def role_user_prompt(role: str, case: Dict[str, Any]) -> str:
    schema_hint = json.dumps(ROLE_MEMO_SCHEMA, ensure_ascii=False)
    return (
        f"ROLE: {role}\n"
        f"DECISION: {case['investment_decision']}\n\n"
        f"FACTS PROVIDED (bounded):\n{json.dumps({k: case[k] for k in case['input_boundary']['allowed_facts_keys'] if k in case}, ensure_ascii=False, indent=2)}\n\n"
        f"OUTPUT REQUIREMENTS:\n"
        f"- Return a JSON object with keys: role, stance, key_arguments, objections, required_conditions, risk_flags\n"
        f"- stance must be one of: APPROVE, REJECT, REVIEW\n"
        f"- Keep arguments grounded in provided synthetic facts and client constraints\n"
        f"- No fabricated sources or market facts\n"
        f"- JSON schema (for your reference): {schema_hint}\n\n"
        f"Now produce the role memo JSON."
    )

SYNTH_SYSTEM = (
    "You are the committee supervisor. "
    "You must preserve dissent. "
    "You must enforce policy: if Compliance/Suitability stance is REJECT, final must be HUMAN_REVIEW or REJECT per configuration. "
    "You must not invent facts. "
    "Return ONLY valid JSON matching the required schema."
)

def synth_user_prompt(case: Dict[str, Any], role_memos: List[Dict[str, Any]]) -> str:
    schema_hint = json.dumps(COMMITTEE_RECORD_SCHEMA, ensure_ascii=False)
    return (
        f"CASE (bounded facts):\n{json.dumps({k: case[k] for k in case['input_boundary']['allowed_facts_keys'] if k in case}, ensure_ascii=False, indent=2)}\n\n"
        f"ROLE MEMOS (JSON):\n{json.dumps(role_memos, ensure_ascii=False, indent=2)}\n\n"
        f"TASK:\n"
        f"1) Create committee_record.role_memos as the provided memos (do not alter meaning).\n"
        f"2) Compute vote_tally counts for APPROVE/REJECT/REVIEW.\n"
        f"3) If there is any disagreement among stances, populate dissent_log with verbatim objections (strings) from minority memos.\n"
        f"4) Provide synthesis_recommendation as a board-ready one-paragraph recommendation.\n"
        f"5) Provide escalation_required.required (boolean) and escalation_required.why.\n"
        f"6) Enforce compliance veto policy.\n\n"
        f"OUTPUT must be JSON matching schema: {schema_hint}\n"
        f"Return ONLY the JSON."
    )

def run_committee(case: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], Optional[Dict[str, Any]], List[Dict[str, Any]]]:
    memos: List[Dict[str, Any]] = []
    local_risks: List[Dict[str, Any]] = []

    for role in Roles:
        obj, raw = call_claude_json(
            system=ROLE_SYSTEM,
            user=role_user_prompt(role, case),
            kind="role_memo",
            role=role,
            max_tokens=CONFIG["max_tokens_role"]
        )
        if obj is None:
            local_risks.append(make_risk(
                severity="HIGH",
                category="ROLE_MEMO_MISSING",
                description=f"Missing/invalid memo for role={role}; using placeholder REVIEW memo.",
                control="Schema enforcement + HUMAN_REVIEW gate."
            ))
            obj = {
                "role": role,
                "stance": "REVIEW",
                "key_arguments": ["Insufficient structured output from model; placeholder used.", "Escalate to human for role assessment."],
                "objections": ["Model output invalid or missing; cannot rely on memo."],
                "required_conditions": ["Human committee member must provide memo."],
                "risk_flags": ["LLM output invalid or missing", "HUMAN_REVIEW required"]
            }

        ok, err = validate_or_none(ROLE_MEMO_SCHEMA, obj)
        if not ok:
            local_risks.append(make_risk(
                severity="HIGH",
                category="ROLE_MEMO_SCHEMA_FAIL",
                description=f"Role memo schema failed for role={role}: {err}",
                control="Force HUMAN_REVIEW; preserve raw response outside artifacts (not stored) and proceed with placeholder."
            ))
            obj = {
                "role": role,
                "stance": "REVIEW",
                "key_arguments": ["Schema invalid; placeholder used.", "Escalate to human for role assessment."],
                "objections": ["Invalid structured memo; cannot rely on content."],
                "required_conditions": ["Human committee member must provide memo."],
                "risk_flags": ["Schema invalid", "HUMAN_REVIEW required"]
            }

        # Hard role name normalization
        obj["role"] = role
        memos.append(obj)

    synth_obj, synth_raw = call_claude_json(
        system=SYNTH_SYSTEM,
        user=synth_user_prompt(case, memos),
        kind="synthesis",
        role="Supervisor",
        max_tokens=CONFIG["max_tokens_synth"]
    )
    if synth_obj is None:
        local_risks.append(make_risk(
            severity="HIGH",
            category="SYNTHESIS_MISSING",
            description="Supervisor synthesis missing/invalid; will build deterministic fallback committee_record.",
            control="Deterministic fallback + HUMAN_REVIEW gate."
        ))
        synth_obj = None

    return memos, synth_obj, local_risks

ROLE_MEMOS, SYNTH_OBJ, LOCAL_RISKS = run_committee(CASE)
for r in LOCAL_RISKS:
    log_risk(r)

print("role_memos:", len(ROLE_MEMOS), "synth_ok:", SYNTH_OBJ is not None)

role_memos: 4 synth_ok: False


##6.GATES

###6.1.OVERVIEW

**CELL 6/10 — Governance Gates, Risk Detection, and Escalation Logic (Pedagogical Explanation)**

This cell is the controls layer. It answers the Board’s primary question: “How do we prevent the system from giving a confident answer when it should not?” The answer is gates—explicit checks that can fail and that trigger escalation.

First, we define Gate A: role completeness. The committee is invalid if not all four roles are present. This is a direct institutional mapping: you would not accept an investment decision that bypassed Compliance or Risk review. If a memo is missing, the gate fails and the system must escalate.

Second, Gate B: dissent preservation. If roles disagree on stance, dissent_log must be non-empty. This is subtle but critical. Organizations often “smooth” disagreement in final write-ups. The dissent gate prevents that by making disagreement an explicit evidence field. If there is disagreement and dissent is empty, the gate fails and triggers HUMAN_REVIEW.

Third, Gate C: policy enforcement. We implement a compliance veto policy: if Compliance/Suitability says REJECT, the final outcome must be HUMAN_REVIEW or REJECT. This reflects real-world constraints: suitability concerns are not “balanced” away by return arguments. The notebook chooses a strict default: compliance veto forces escalation. This is exactly the type of rule a Board should approve as policy.

The cell also addresses schema failure at the committee record level. If the supervisor’s output fails validation, the system uses a deterministic fallback committee record and escalates. This design prevents failures in the AI layer from corrupting the governance record. Even when the AI fails, the system still produces a trace and a risk log that explains the failure.

Finally, this cell computes a preliminary decision. If any HIGH-severity governance risk exists, the outcome becomes HUMAN_REVIEW by default. This is conservative by design. It is better to escalate than to output a false sense of safety.

In board terms: Cell 6 is the risk control mechanism that keeps AI useful without allowing it to bypass institutional safeguards. It converts technical failures and policy conflicts into explicit, reviewable escalation signals.

###6.2.CODE AND IMPLEMENTATION

In [6]:
# CELL 6/10 — Gates + risk detection + escalation logic
def gate_role_completeness(role_memos: List[Dict[str, Any]]) -> Tuple[bool, Optional[Dict[str, Any]]]:
    roles_present = {m.get("role") for m in role_memos}
    missing = [r for r in Roles if r not in roles_present]
    if missing:
        return False, make_risk(
            severity="HIGH",
            category="GATE_ROLE_COMPLETENESS_FAIL",
            description=f"Missing role memos for: {missing}",
            control="Gate A requires all 4 roles; set HUMAN_REVIEW."
        )
    return True, None

def gate_policy_enforcement(role_memos: List[Dict[str, Any]], synth_obj: Optional[Dict[str, Any]]) -> Tuple[bool, Optional[Dict[str, Any]], str]:
    compliance = next((m for m in role_memos if m.get("role") == "Compliance/Suitability"), None)
    if not compliance:
        return False, make_risk(
            severity="HIGH",
            category="GATE_COMPLIANCE_MISSING",
            description="Compliance/Suitability memo missing; cannot enforce veto.",
            control="Force HUMAN_REVIEW."
        ), "HUMAN_REVIEW"
    if CONFIG["policy"]["compliance_veto"] and compliance.get("stance") == "REJECT":
        # Strict policy: veto triggers HUMAN_REVIEW (or REJECT)
        outcome = CONFIG["policy"]["compliance_veto_outcome"]
        return True, make_risk(
            severity="HIGH",
            category="POLICY_COMPLIANCE_VETO",
            description=f"Compliance/Suitability stance=REJECT; policy forces outcome={outcome}.",
            control="Compliance veto rule applied; escalate to HUMAN_REVIEW / REJECT."
        ), outcome
    return True, None, ""  # no forced override

def gate_dissent_preservation(role_memos: List[Dict[str, Any]], committee_record: Dict[str, Any]) -> Tuple[bool, Optional[Dict[str, Any]]]:
    stances = [m.get("stance") for m in role_memos]
    disagreement = len(set(stances)) > 1
    dissent_log = committee_record.get("committee_record", {}).get("dissent_log", [])
    if disagreement and (not isinstance(dissent_log, list) or len(dissent_log) == 0):
        return False, make_risk(
            severity="HIGH",
            category="GATE_DISSENT_PRESERVATION_FAIL",
            description="Disagreement exists but dissent_log is empty.",
            control="Gate B requires dissent preservation; force HUMAN_REVIEW."
        )
    return True, None

def deterministic_fallback_committee_record(role_memos: List[Dict[str, Any]]) -> Dict[str, Any]:
    tally = {"APPROVE": 0, "REJECT": 0, "REVIEW": 0}
    for m in role_memos:
        s = m.get("stance", "REVIEW")
        if s not in tally:
            s = "REVIEW"
        tally[s] += 1
    stances = [m.get("stance","REVIEW") for m in role_memos]
    majority = max(tally.items(), key=lambda kv: kv[1])[0]
    dissent = []
    if len(set(stances)) > 1:
        for m in role_memos:
            if m.get("stance") != majority:
                for obj in (m.get("objections") or []):
                    dissent.append(str(obj))
    return {
        "committee_record": {
            "role_memos": role_memos,
            "vote_tally": tally,
            "dissent_log": dissent[:30],
            "synthesis_recommendation": "Deterministic fallback synthesis used due to invalid/missing supervisor output. Escalate to human committee.",
            "escalation_required": {"required": True, "why": "Supervisor synthesis invalid/missing; governance requires human review."}
        }
    }

# Build / validate committee_record
if SYNTH_OBJ is None:
    COMMITTEE_OBJ = deterministic_fallback_committee_record(ROLE_MEMOS)
else:
    ok, err = validate_or_none(COMMITTEE_RECORD_SCHEMA, SYNTH_OBJ)
    if not ok:
        log_risk(make_risk(
            severity="HIGH",
            category="COMMITTEE_RECORD_SCHEMA_FAIL",
            description=f"Supervisor output failed committee_record schema: {err}",
            control="Use deterministic fallback + force HUMAN_REVIEW."
        ))
        COMMITTEE_OBJ = deterministic_fallback_committee_record(ROLE_MEMOS)
    else:
        COMMITTEE_OBJ = SYNTH_OBJ

# Gate A
okA, riskA = gate_role_completeness(ROLE_MEMOS)
if not okA and riskA: log_risk(riskA)

# Gate B
okB, riskB = gate_dissent_preservation(ROLE_MEMOS, COMMITTEE_OBJ)
if not okB and riskB: log_risk(riskB)

# Gate C (policy enforcement)
okC, riskC, forced_outcome = gate_policy_enforcement(ROLE_MEMOS, COMMITTEE_OBJ)
if riskC: log_risk(riskC)

# Persist risk log
write_json(RISK_LOG_PATH, {"run_id": RUN_ID, "timestamp_utc": utc_now_iso(), "risks": _risks})

# Determine preliminary decision
PRELIM_DECISION = "HUMAN_REVIEW" if any(r["severity"] == "HIGH" for r in _risks) else "APPROVE"
if forced_outcome:
    PRELIM_DECISION = forced_outcome

print("prelim_decision:", PRELIM_DECISION, "high_risks:", sum(1 for r in _risks if r["severity"]=="HIGH"))

prelim_decision: HUMAN_REVIEW high_risks: 5


##7.TRACE BUILDER

###7.1.OVERVIEW

**CELL 7/10 — Trace Builder: Creating reasoning_trace.json (Pedagogical Explanation)**

This cell builds the primary artifact the committee will care about: reasoning_trace.json. The trace is the structured record of what happened. It is not a narrative; it is an object designed for audit and review.

The trace includes: the run_id, timestamp, the reasoning pattern (COMMITTEE), the input boundary, the complete committee record (role memos, vote tally, dissent log, synthesis, escalation), the gate results, the decision, and a snapshot of risks. This is effectively “minutes + evidence” in machine-readable form.

A key step in this cell is normalization. We reorder role memos to a fixed role order. That sounds minor, but it matters for governance. Fixed ordering prevents accidental role omission and makes comparisons across runs easy. If a role is missing, we insert a placeholder to maintain structural integrity while still capturing the governance failure. This is how we prevent silent structural drift.

We also ensure the committee record is exactly four memos. In a board setting, “almost complete” is not complete. The trace must preserve the quorum requirement.

The trace also records gate outcomes: whether role completeness passed, whether dissent preservation passed, whether policy enforcement passed, and whether there was a forced outcome. This is crucial: it explains why the decision is what it is. Without this, a reader might see “HUMAN_REVIEW” and not know whether it came from disagreement, a compliance veto, or a schema failure.

Finally, we write the trace to artifacts/reasoning_trace.json. This file becomes the canonical reference object for downstream reporting, explanation generation, and archiving. It is also the right object to show to auditors because it is explicit, bounded, and systematically produced.

In board terms: Cell 7 creates the “evidence spine.” It’s the record that lets the institution demonstrate, later, that it followed policy: the right roles participated, dissent was captured, gates were enforced, and escalation occurred when required.

###7.2.CODE AND IMPLEMENTATION

In [7]:
# CELL 7/10 — Trace builder (reasoning_trace.json) + normalization
def build_reasoning_trace(case: Dict[str, Any], committee_obj: Dict[str, Any], risks: List[Dict[str, Any]], decision: str) -> Dict[str, Any]:
    cr = committee_obj.get("committee_record", {})
    # Normalize role order
    memos_by_role = {m.get("role"): m for m in cr.get("role_memos", [])}
    ordered_memos = [memos_by_role.get(r) for r in Roles if r in memos_by_role]
    # Ensure exactly 4 entries (pad with deterministic placeholders if needed)
    while len(ordered_memos) < 4:
        missing_role = Roles[len(ordered_memos)]
        ordered_memos.append({
            "role": missing_role,
            "stance": "REVIEW",
            "key_arguments": ["Missing memo; placeholder used."],
            "objections": ["Missing memo."],
            "required_conditions": ["Human memo required."],
            "risk_flags": ["Missing memo", "HUMAN_REVIEW required"]
        })
    ordered_memos = ordered_memos[:4]

    vote_tally = cr.get("vote_tally", {"APPROVE": 0, "REJECT": 0, "REVIEW": 0})
    dissent_log = cr.get("dissent_log", [])

    trace = {
        "run_id": RUN_ID,
        "timestamp_utc": utc_now_iso(),
        "pattern": "COMMITTEE",
        "input_boundary": case.get("input_boundary", {}),
        "committee_record": {
            "role_memos": ordered_memos,
            "vote_tally": vote_tally,
            "dissent_log": dissent_log,
            "synthesis_recommendation": cr.get("synthesis_recommendation", ""),
            "escalation_required": cr.get("escalation_required", {"required": True, "why": "Missing escalation info"})
        },
        "gates": {
            "GateA_role_completeness": okA,
            "GateB_dissent_preservation": okB,
            "GateC_policy_enforcement": okC,
            "forced_outcome": forced_outcome or ""
        },
        "decision": decision,
        "risks_snapshot": risks
    }
    return trace

TRACE = build_reasoning_trace(CASE, COMMITTEE_OBJ, _risks, PRELIM_DECISION)
write_json(TRACE_PATH, TRACE)
print("trace_written:", TRACE_PATH)

trace_written: artifacts/reasoning_trace.json


##8.REPORT COMPOSER

###8.1.OVERVIEW

**CELL 8/10 — Final Report Composer with Strict Schema (Pedagogical Explanation)**

This cell converts the trace into a board-facing final_report.json. The final report is designed to be readable, but it is still controlled by schema so it cannot quietly violate governance requirements.

The report includes: executive summary, facts_provided (verbatim bounded case packet), assumptions_introduced, open_items, questions_to_verify, analysis, draft_output, committee_summary, final_decision, confidence, and verification_status = “Not verified.” The separation of these fields is a governance feature. It prevents the report from mixing facts and assumptions in a way that is hard to detect.

We treat role “required_conditions” as open items and as questions to verify. This is a disciplined mapping: required conditions represent missing evidence that would change a stance. Turning them into explicit questions makes the output operational. A committee can assign owners to those questions.

We also include explicit assumptions stating that regime and basket characteristics are synthetic proxies and must not be treated as real estimates. This avoids the subtle failure mode where a reader forgets the synthetic nature and interprets the analysis as a real market view.

We compute confidence in a conservative way: if the decision is HUMAN_REVIEW, confidence is low. If there are high risks, confidence is low. If all roles approve and there are no high risks, confidence can be higher. This is not meant as a statistical confidence; it is a governance confidence reflecting process integrity.

The cell validates the final report against a strict JSON schema. If validation fails, it logs a high-severity risk and forces HUMAN_REVIEW. As a last resort, it produces a minimal schema-compliant report stating that the output must not be used.

In board terms: Cell 8 is how we present controlled output. It is not trying to impress; it is trying to be safe. It ensures that even a polished narrative cannot escape the required structure: facts vs assumptions vs open items, with explicit non-verification.

###8.2.CODE AND IMPLEMENTATION

In [8]:
# CELL 8/10 — Report composer (final_report.json) with strict schema
def stance_to_confidence(stances: List[str], decision: str, high_risks: int) -> str:
    if decision in ("HUMAN_REVIEW",):
        return "low"
    if high_risks > 0:
        return "low"
    if len(set(stances)) == 1 and stances[0] == "APPROVE":
        return "high"
    if "REJECT" in stances:
        return "medium"
    return "medium"

def build_final_report(case: Dict[str, Any], trace: Dict[str, Any], risks: List[Dict[str, Any]], final_decision: str) -> Dict[str, Any]:
    cr = trace["committee_record"]
    memos = cr["role_memos"]
    stances = [m.get("stance","REVIEW") for m in memos]
    high_risks = sum(1 for r in risks if r["severity"] == "HIGH")

    facts_provided = {k: case[k] for k in case["input_boundary"]["allowed_facts_keys"] if k in case}
    assumptions_introduced: List[str] = []
    open_items: List[str] = []
    questions: List[str] = []

    # Treat all "required_conditions" as open items/questions (governance-first)
    for m in memos:
        for c in (m.get("required_conditions") or []):
            s = str(c).strip()
            if s:
                open_items.append(f"[{m.get('role')}] {s}")
                questions.append(f"What evidence would satisfy: {s} (owner: {m.get('role')})")

    # Always include explicit non-verification + synthetic nature
    assumptions_introduced.extend([
        "Basket characteristics are synthetic proxies and not market estimates.",
        "Regime indicator is synthetic and illustrative only.",
        "Any implied risk/return tradeoffs are qualitative and require validation."
    ])

    dissent = cr.get("dissent_log", [])
    policy_notes = []
    compliance = next((m for m in memos if m.get("role") == "Compliance/Suitability"), None)
    if compliance and compliance.get("stance") == "REJECT":
        policy_notes.append("Compliance veto triggered: Compliance/Suitability stance=REJECT.")
        policy_notes.append(f"Policy outcome enforced: {CONFIG['policy']['compliance_veto_outcome']}.")

    if high_risks > 0 and "HUMAN_REVIEW" not in policy_notes:
        policy_notes.append("High-severity governance risks present; outcome escalated accordingly.")

    per_role_summaries = []
    for m in memos:
        per_role_summaries.append({
            "role": m.get("role"),
            "stance": m.get("stance"),
            "top_arguments": (m.get("key_arguments") or [])[:3],
            "top_objections": (m.get("objections") or [])[:3],
            "risk_flags": (m.get("risk_flags") or [])[:5]
        })

    exec_summary = (
        "This is a simulated investment committee record for a thematic equity basket under regime uncertainty. "
        "Inputs are synthetic and bounded; no external market facts are used. "
        f"Outcome: {final_decision}. "
        "Decision is governed by role completeness, dissent preservation, and compliance policy."
    )

    analysis_text = (
        f"Client constraints indicate risk tolerance '{case['client_constraints']['risk_tolerance_band']}' with stated "
        f"max drawdown tolerance of {case['client_constraints']['max_drawdown_tolerance_pct_stated']}%. "
        f"Regime signal is '{case['regime_indicator']['current_regime']}' with {case['regime_indicator']['confidence']} confidence (synthetic). "
        f"Basket theme is '{case['basket_characteristics']['theme']}', with synthetic volatility proxy "
        f"{case['basket_characteristics']['volatility_annualized_pct']}% and correlation proxy '{case['basket_characteristics']['correlation_proxy']}'. "
        "Committee memos should be interpreted as structured viewpoints with explicit conditions; they are not verified facts."
    )

    draft_output = (
        "Draft action: If proceeding, constrain position sizing to match stated drawdown tolerance, "
        "document liquidity expectations, and require validation of basket construction, concentration, and scenario stress. "
        "If not proceeding, document which role conditions were unmet and assign owners to resolve open items."
    )

    report = {
        "run_id": RUN_ID,
        "timestamp_utc": utc_now_iso(),
        "executive_summary": exec_summary,
        "facts_provided": facts_provided,
        "assumptions_introduced": sorted(list(dict.fromkeys(assumptions_introduced))),
        "open_items": sorted(list(dict.fromkeys(open_items)))[:50],
        "questions_to_verify": sorted(list(dict.fromkeys(questions)))[:50],
        "analysis": analysis_text,
        "draft_output": draft_output,
        "committee_summary": {
            "per_role_summaries": per_role_summaries,
            "vote_tally": cr.get("vote_tally", {}),
            "dissent": dissent,
            "policy_notes": policy_notes
        },
        "final_decision": final_decision,
        "confidence": stance_to_confidence(stances, final_decision, high_risks),
        "verification_status": "Not verified"
    }
    return report

FINAL_REPORT = build_final_report(CASE, TRACE, _risks, PRELIM_DECISION)

okR, errR = validate_or_none(FINAL_REPORT_SCHEMA, FINAL_REPORT)
if not okR:
    log_risk(make_risk(
        severity="HIGH",
        category="FINAL_REPORT_SCHEMA_FAIL",
        description=f"Final report schema invalid: {errR}",
        control="Force HUMAN_REVIEW; write best-effort report with explicit failure."
    ))
    FINAL_REPORT["final_decision"] = "HUMAN_REVIEW"
    FINAL_REPORT["confidence"] = "low"
    FINAL_REPORT["analysis"] += " | NOTE: Final report failed schema validation; governance escalation required."
    # Re-validate (best effort)
    okR2, errR2 = validate_or_none(FINAL_REPORT_SCHEMA, FINAL_REPORT)
    if not okR2:
        # As last resort, keep minimal schema-compliant content
        FINAL_REPORT = {
            "run_id": RUN_ID,
            "timestamp_utc": utc_now_iso(),
            "executive_summary": "Schema failure occurred; escalate to HUMAN_REVIEW.",
            "facts_provided": {k: CASE[k] for k in CASE["input_boundary"]["allowed_facts_keys"] if k in CASE},
            "assumptions_introduced": ["Schema failure; output minimized."],
            "open_items": ["Human reviewer to regenerate final report."],
            "questions_to_verify": ["What caused schema failure?"],
            "analysis": "Minimized due to schema failure.",
            "draft_output": "Do not use this output for decisions; HUMAN_REVIEW required.",
            "committee_summary": {
                "per_role_summaries": [],
                "vote_tally": {},
                "dissent": [],
                "policy_notes": ["Schema failure"]
            },
            "final_decision": "HUMAN_REVIEW",
            "confidence": "low",
            "verification_status": "Not verified"
        }

write_json(FINAL_REPORT_PATH, FINAL_REPORT)
write_json(RISK_LOG_PATH, {"run_id": RUN_ID, "timestamp_utc": utc_now_iso(), "risks": _risks})
print("final_report_written:", FINAL_REPORT_PATH, "schema_ok:", okR)

final_report_written: artifacts/final_report.json schema_ok: True


##9.EXECUTION

###9.1.0VERVIEW

**CELL 9/10 — Run Orchestrator and Manifest Finalization (Pedagogical Explanation)**

This cell closes the loop on operational governance. It updates the run manifest with completion time, output paths, the final decision, and risk counts. This is the step that turns a notebook run into a traceable “job” with a start identity and an end identity.

The manifest is the anchor artifact for operations. If you store these bundles over time, the manifest becomes the index: it lets you search for a run by run_id, see what model and config were used, and retrieve the exact files. That matters for audit, incident response, and quality monitoring.

The cell writes a completed_timestamp_utc. This seems small, but it matters. In regulated environments, time matters: when was a recommendation produced, relative to market events, policy updates, or client changes? Using UTC ensures consistent time interpretation.

It also records output paths. This is practical governance. A Board packet should not require manual hunting through a notebook. The manifest should tell you exactly where the evidence lives.

Finally, it summarizes risk counts. This is the first step toward governance dashboards. Over many runs, you can track: how often do schema failures occur, how often does compliance veto trigger, how often do we escalate. Those metrics become operational controls.

In board terms: Cell 9 makes the run auditable as a unit of work. It is a transaction record. Without it, the notebook is just an interactive document. With it, the notebook becomes an institutional pipeline run that can be logged, archived, compared, and reviewed.

###9.2.CODE AND IMPLEMENTATION

In [9]:
# CELL 9/10 — Run orchestrator: executes pipeline end-to-end, writes artifacts
def orchestrate() -> Dict[str, Any]:
    manifest = json.load(open("artifacts/run_manifest.json","r",encoding="utf-8"))
    manifest["completed_timestamp_utc"] = utc_now_iso()
    manifest["outputs"] = {
        "prompts_log_jsonl": PROMPTS_LOG_PATH,
        "reasoning_trace_json": TRACE_PATH,
        "risk_log_json": RISK_LOG_PATH,
        "final_report_json": FINAL_REPORT_PATH
    }
    manifest["decision"] = FINAL_REPORT.get("final_decision","HUMAN_REVIEW")
    manifest["risk_counts"] = {
        "LOW": sum(1 for r in _risks if r["severity"]=="LOW"),
        "MEDIUM": sum(1 for r in _risks if r["severity"]=="MEDIUM"),
        "HIGH": sum(1 for r in _risks if r["severity"]=="HIGH")
    }
    write_json("artifacts/run_manifest.json", manifest)
    return manifest

MANIFEST = orchestrate()
print(json.dumps(MANIFEST, indent=2, ensure_ascii=False)[:1200])

{
  "config_hash": "f481e43b1f1f9f452fa80f4d1493476286fefef03fc81b73aaf905c842ef9cce",
  "determinism": {
    "PYTHONHASHSEED": "0",
    "note": "Notebook-level best-effort determinism.",
    "seed": 1337
  },
  "inputs": {
    "case_id": "SYN-COMMITTEE-1337",
    "input_boundary": {
      "allowed_facts_keys": [
        "investment_decision",
        "client_constraints",
        "regime_indicator",
        "basket_characteristics"
      ],
      "prohibited": [
        "real tickers",
        "real market returns",
        "real vol estimates",
        "forward-looking guarantees",
        "external sources not provided"
      ]
    }
  },
  "model": "claude-haiku-4-5-20251001",
  "notebook": "Notebook 4 — Committee Reasoning",
  "pattern": "COMMITTEE",
  "run_id": "7492e5e00d94ca0b",
  "timestamp_utc": "2026-02-20T20:22:01.974181+00:00",
  "completed_timestamp_utc": "2026-02-20T20:24:57.076974+00:00",
  "outputs": {
    "prompts_log_jsonl": "artifacts/prompts_log.jsonl",
    "reason

##10.AUDIT BUNDLE

###10.1.OVERVIEW

**CELL 10/10 — Packaging Deliverables and Console Summary (Pedagogical Explanation)**

This final cell packages the complete evidence bundle into deliverables/deliverables.zip and prints a concise summary of where everything is stored. It is the operational handoff layer: it produces a single file that can be distributed to committee members, archived, or attached to a governance system.

The zip includes all artifacts (manifest, prompts log, reasoning trace, risk log, final report). Packaging matters because governance is often organizational, not technical: people email files, upload to portals, store in document management systems. A single deliverables.zip reduces the risk of missing components. If you send only the final report without the trace and risk log, you lose the audit context.

The cell also includes a copy of final_report.json at the top level of the zip for convenience. This is a usability feature: reviewers can read the report without navigating folder structures, but auditors still have the deeper evidence.

Finally, the cell prints a summary: run_id, UTC timestamp, file paths, final decision, confidence, and risk counts. This is not just “nice output.” It is a run receipt. If this notebook is used in a broader workflow, that summary can be captured by an orchestrator (or copied into minutes) to reference the exact artifact bundle used.

In board terms: Cell 10 is how we make the output usable and reviewable in the real world. A reasoning pipeline is only valuable if it produces evidence in a form that can be shared, archived, and inspected. This cell ensures the notebook ends not with a paragraph, but with an auditable package.

If we later integrate this notebook into enterprise systems, this packaging step maps naturally to uploading artifacts to controlled storage, generating immutable links, and registering the run in a governance registry.

###10.2.CODE AND IMPLEMENTATION

In [10]:
# CELL 10/10 — Packaging: zip deliverables + minimal console summary of outputs/paths
ZIP_PATH = DELIVERABLES_DIR / "deliverables.zip"

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in ARTIFACTS_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(p))
    # Include a copy of the final report at top-level inside zip for convenience
    z.write(FINAL_REPORT_PATH, arcname="final_report.json")

summary = {
    "run_id": RUN_ID,
    "timestamp_utc": utc_now_iso(),
    "paths": {
        "run_manifest": str(ARTIFACTS_DIR / "run_manifest.json"),
        "prompts_log": str(ARTIFACTS_DIR / "prompts_log.jsonl"),
        "reasoning_trace": str(ARTIFACTS_DIR / "reasoning_trace.json"),
        "risk_log": str(ARTIFACTS_DIR / "risk_log.json"),
        "final_report": str(ARTIFACTS_DIR / "final_report.json"),
        "deliverables_zip": str(ZIP_PATH)
    },
    "final_decision": FINAL_REPORT.get("final_decision"),
    "confidence": FINAL_REPORT.get("confidence"),
    "risk_counts": {
        "LOW": sum(1 for r in _risks if r["severity"]=="LOW"),
        "MEDIUM": sum(1 for r in _risks if r["severity"]=="MEDIUM"),
        "HIGH": sum(1 for r in _risks if r["severity"]=="HIGH")
    }
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "run_id": "7492e5e00d94ca0b",
  "timestamp_utc": "2026-02-20T20:25:25.845730+00:00",
  "paths": {
    "run_manifest": "artifacts/run_manifest.json",
    "prompts_log": "artifacts/prompts_log.jsonl",
    "reasoning_trace": "artifacts/reasoning_trace.json",
    "risk_log": "artifacts/risk_log.json",
    "final_report": "artifacts/final_report.json",
    "deliverables_zip": "deliverables/deliverables.zip"
  },
  "final_decision": "HUMAN_REVIEW",
  "confidence": "low",
  "risk_counts": {
    "LOW": 0,
    "MEDIUM": 0,
    "HIGH": 5
  }
}


In [13]:
# DROP-IN REPLACEMENT CELL — Robust Trace Explanation (handles non-JSON output safely)
# Fixes: Claude sometimes returns prose; this cell enforces JSON-only output + has a deterministic repair path.

import json, pathlib, hashlib, datetime, re
from typing import Any, Dict, Optional, Tuple, List
from google.colab import userdata
from anthropic import Anthropic
from jsonschema import Draft202012Validator, ValidationError

ARTIFACTS_DIR = pathlib.Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "claude-haiku-4-5-20251001"

def utc_now_iso() -> str:
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def write_text(path: str, text: str) -> None:
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("w", encoding="utf-8") as f:
        f.write(text)

def write_json(path: str, obj: Any) -> None:
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, sort_keys=True)

def append_jsonl(path: str, obj: Any) -> None:
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, sort_keys=True) + "\n")

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

_RE_EMAIL = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
_RE_PHONE = re.compile(r"\b(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{2,4}\)?[-.\s]?)?\d{3,4}[-.\s]?\d{4}\b")
_RE_KEYLIKE = re.compile(r"\b(sk-[A-Za-z0-9]{8,}|ANTHROPIC_[A-Za-z0-9_]{8,}|[A-Za-z0-9_-]{24,})\b")

def redact_text(s: str) -> str:
    s2 = _RE_EMAIL.sub("[REDACTED_EMAIL]", s)
    s2 = _RE_PHONE.sub("[REDACTED_PHONE]", s2)
    s2 = _RE_KEYLIKE.sub("[REDACTED_SECRET]", s2)
    return s2

def validate(schema: Dict[str, Any], obj: Any) -> Tuple[bool, Optional[str]]:
    try:
        Draft202012Validator(schema).validate(obj)
        return True, None
    except ValidationError as e:
        return False, str(e)

def extract_json_candidates(text: str) -> List[str]:
    # Finds balanced-ish JSON object candidates by scanning for braces.
    # Not perfect, but robust enough for typical LLM outputs.
    candidates = []
    stack = 0
    start = None
    for i, ch in enumerate(text):
        if ch == "{":
            if stack == 0:
                start = i
            stack += 1
        elif ch == "}":
            if stack > 0:
                stack -= 1
                if stack == 0 and start is not None:
                    candidates.append(text[start:i+1])
                    start = None
    # Prefer larger candidates first (usually the full object)
    candidates.sort(key=len, reverse=True)
    return candidates

def try_parse_any_json(text: str) -> Optional[Dict[str, Any]]:
    # 1) direct
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    # 2) candidates from brace scan
    for cand in extract_json_candidates(text):
        try:
            obj = json.loads(cand)
            if isinstance(obj, dict):
                return obj
        except Exception:
            continue
    return None

def minimal_fallback_report(run_id: str, facts: Dict[str, Any]) -> Dict[str, Any]:
    # Deterministic, schema-aligned minimal output for governance safety
    return {
        "run_id": run_id,
        "timestamp_utc": utc_now_iso(),
        "verification_status": "Not verified",
        "executive_summary_plain": (
            "The trace explanation could not be generated as valid structured JSON from the model output. "
            "This is treated as a governance failure and requires HUMAN_REVIEW. "
            "A minimal explanation is provided to preserve process continuity."
        ),
        "what_inputs_were_used_plain": (
            "Inputs were taken only from the notebook artifacts (run_manifest, reasoning_trace, risk_log, final_report). "
            "No external market data, tickers, or sources were used."
        ),
        "how_the_committee_process_worked_plain": (
            "The pipeline requested one structured memo per role (Portfolio, Risk, Compliance/Suitability, Macro). "
            "A supervisor then synthesized a vote tally, dissent log, and escalation recommendation. "
            "Gates enforced role completeness, dissent preservation, and compliance policy rules."
        ),
        "role_by_role_walkthrough_plain": (
            "Role-by-role details could not be reliably reconstructed into the required schema from the model output. "
            "Use artifacts/reasoning_trace.json directly for the canonical per-role memos."
        ),
        "vote_and_policy_plain": (
            "The final decision must follow the policy gates recorded in the trace. "
            "If Compliance/Suitability stance is REJECT, policy requires escalation (HUMAN_REVIEW or REJECT)."
        ),
        "dissent_and_objections_plain": (
            "Dissent must be preserved when roles disagree. "
            "Consult artifacts/reasoning_trace.json for the dissent_log field."
        ),
        "gates_and_controls_plain": (
            "Gates are recorded in the trace: role completeness, dissent preservation, and policy enforcement. "
            "If any gate fails or if high-severity risks exist, the run should escalate to HUMAN_REVIEW."
        ),
        "what_is_open_and_needs_verification_plain": (
            "Open items and questions to verify are listed in artifacts/final_report.json and must be assigned owners. "
            "Verification status remains Not verified until humans validate assumptions and conditions."
        ),
        "what_we_did_not_do_plain": (
            "This pipeline does not validate real market data, does not forecast returns, and does not provide fiduciary advice. "
            "It produces a governed reasoning record and escalates uncertainty for human decision-makers."
        ),
        "glossary_plain": (
            "Trace: the structured record of role memos, votes, dissent, gates, and decision.\n"
            "Gate: a control check that can force escalation.\n"
            "Dissent: minority objections preserved verbatim.\n"
            "Not verified: output must be validated by humans before use."
        ),
        "facts_provided": facts,
        "assumptions_introduced": ["Model output not parsed; assumptions require human review."],
        "open_items": ["HUMAN_REVIEW: regenerate trace explanation with stricter output control or repair step."],
        "questions_to_verify": ["Why did the model fail to return valid JSON under JSON-only instructions?"]
    }

# Load artifacts
trace_path = ARTIFACTS_DIR / "reasoning_trace.json"
final_report_path = ARTIFACTS_DIR / "final_report.json"
manifest_path = ARTIFACTS_DIR / "run_manifest.json"
risk_log_path = ARTIFACTS_DIR / "risk_log.json"

for p in [trace_path, final_report_path, manifest_path, risk_log_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required artifact: {p}")

TRACE = json.load(open(trace_path, "r", encoding="utf-8"))
FINAL = json.load(open(final_report_path, "r", encoding="utf-8"))
MANIFEST = json.load(open(manifest_path, "r", encoding="utf-8"))
RISKS = json.load(open(risk_log_path, "r", encoding="utf-8"))

RUN_ID = TRACE.get("run_id") or MANIFEST.get("run_id") or "UNKNOWN"
FACTS = FINAL.get("facts_provided") or {}

# Schema for explanation report (strict)
TRACE_EXPLAIN_SCHEMA: Dict[str, Any] = {
  "type": "object",
  "additionalProperties": False,
  "required": [
    "run_id","timestamp_utc","verification_status",
    "executive_summary_plain","what_inputs_were_used_plain",
    "how_the_committee_process_worked_plain",
    "role_by_role_walkthrough_plain",
    "vote_and_policy_plain",
    "dissent_and_objections_plain",
    "gates_and_controls_plain",
    "what_is_open_and_needs_verification_plain",
    "what_we_did_not_do_plain",
    "glossary_plain",
    "facts_provided",
    "assumptions_introduced",
    "open_items",
    "questions_to_verify"
  ],
  "properties": {
    "run_id": {"type":"string"},
    "timestamp_utc": {"type":"string"},
    "verification_status": {"type":"string", "enum":["Not verified"]},
    "executive_summary_plain": {"type":"string"},
    "what_inputs_were_used_plain": {"type":"string"},
    "how_the_committee_process_worked_plain": {"type":"string"},
    "role_by_role_walkthrough_plain": {"type":"string"},
    "vote_and_policy_plain": {"type":"string"},
    "dissent_and_objections_plain": {"type":"string"},
    "gates_and_controls_plain": {"type":"string"},
    "what_is_open_and_needs_verification_plain": {"type":"string"},
    "what_we_did_not_do_plain": {"type":"string"},
    "glossary_plain": {"type":"string"},
    "facts_provided": {"type":"object"},
    "assumptions_introduced": {"type":"array", "items":{"type":"string"}},
    "open_items": {"type":"array", "items":{"type":"string"}},
    "questions_to_verify": {"type":"array", "items":{"type":"string"}}
  }
}

API_KEY = userdata.get("ANTHROPIC_API_KEY")
if not API_KEY or not isinstance(API_KEY, str):
    raise RuntimeError("Missing ANTHROPIC_API_KEY in Colab Secrets (google.colab.userdata).")

client = Anthropic(api_key=API_KEY)

# Stronger instruction: return JSON only, plus explicit "BEGIN_JSON" / "END_JSON" guard
SYSTEM = (
    "You write for a Board of Directors. Use plain language and be pedagogical. "
    "ABSOLUTE RULES: "
    "1) Return ONLY JSON. No prose outside JSON. "
    "2) Do NOT invent facts; only use the provided artifacts. "
    "3) Preserve dissent by copying objections verbatim from the artifacts. "
    "4) verification_status must be exactly 'Not verified'. "
)

USER = (
    "Return ONE JSON object that validates against this exact schema:\n"
    f"{json.dumps(TRACE_EXPLAIN_SCHEMA, ensure_ascii=False)}\n\n"
    "Fill every required field. Use plain language. Use the artifacts only.\n\n"
    "ARTIFACTS:\n"
    f"run_manifest:\n{json.dumps(MANIFEST, ensure_ascii=False, indent=2)}\n\n"
    f"reasoning_trace:\n{json.dumps(TRACE, ensure_ascii=False, indent=2)}\n\n"
    f"risk_log:\n{json.dumps(RISKS, ensure_ascii=False, indent=2)}\n\n"
    f"final_report:\n{json.dumps(FINAL, ensure_ascii=False, indent=2)}\n"
)

# Log prompt (redacted + hashed)
prompt_for_log = f"SYSTEM:\n{SYSTEM}\n\nUSER:\n{USER}"
append_jsonl("artifacts/prompts_log.jsonl", {
    "run_id": RUN_ID,
    "timestamp_utc": utc_now_iso(),
    "kind": "trace_explanation_v2",
    "role": "Committee-Explainer",
    "model": MODEL_NAME,
    "prompt_sha256": sha256_text(prompt_for_log),
    "redacted_prompt": redact_text(prompt_for_log)[:2000]
})

# First attempt
msg = client.messages.create(
    model=MODEL_NAME,
    max_tokens=2200,
    temperature=0.0,
    system=SYSTEM,
    messages=[{"role": "user", "content": USER}],
)

raw_text = ""
if hasattr(msg, "content") and msg.content:
    raw_text = "".join([getattr(b, "text", "") for b in msg.content])
else:
    raw_text = str(msg)

obj = try_parse_any_json(raw_text)

# If parsing failed, run a second "repair" call: feed raw output and ask to convert to valid JSON
if obj is None:
    append_jsonl("artifacts/prompts_log.jsonl", {
        "run_id": RUN_ID,
        "timestamp_utc": utc_now_iso(),
        "kind": "trace_explanation_repair_prompt",
        "role": "Committee-Explainer-Repair",
        "model": MODEL_NAME,
        "prompt_sha256": sha256_text(raw_text),
        "redacted_prompt": redact_text(raw_text)[:1200]
    })

    REPAIR_SYSTEM = (
        "You are a JSON repair tool. Convert the given content into a SINGLE valid JSON object "
        "that matches the provided schema. Return JSON ONLY."
    )
    REPAIR_USER = (
        "SCHEMA:\n"
        f"{json.dumps(TRACE_EXPLAIN_SCHEMA, ensure_ascii=False)}\n\n"
        "CONTENT TO CONVERT (may include prose):\n"
        f"{raw_text}\n\n"
        "Return ONLY valid JSON matching the schema. Do not invent facts."
    )

    msg2 = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2200,
        temperature=0.0,
        system=REPAIR_SYSTEM,
        messages=[{"role":"user","content": REPAIR_USER}],
    )

    raw2 = ""
    if hasattr(msg2, "content") and msg2.content:
        raw2 = "".join([getattr(b, "text", "") for b in msg2.content])
    else:
        raw2 = str(msg2)

    obj = try_parse_any_json(raw2)
    raw_text = raw2 if obj is not None else raw_text

# Validate; if still invalid, fall back deterministically (governance-first)
if obj is None:
    write_text("artifacts/trace_explained_report_raw.txt", raw_text)
    obj = minimal_fallback_report(RUN_ID, FACTS)

ok, err = validate(TRACE_EXPLAIN_SCHEMA, obj)
if not ok:
    # last resort: write raw + fallback
    write_text("artifacts/trace_explained_report_raw.txt", raw_text)
    obj = minimal_fallback_report(RUN_ID, FACTS)

# Persist outputs
write_json("artifacts/trace_explained_report.json", obj)

# Write plain text version for sharing
narrative = (
    f"RUN ID: {obj['run_id']}\n"
    f"TIMESTAMP (UTC): {obj['timestamp_utc']}\n"
    f"VERIFICATION STATUS: {obj['verification_status']}\n\n"
    "EXECUTIVE SUMMARY\n"
    f"{obj['executive_summary_plain']}\n\n"
    "WHAT INPUTS WERE USED\n"
    f"{obj['what_inputs_were_used_plain']}\n\n"
    "HOW THE COMMITTEE PROCESS WORKED\n"
    f"{obj['how_the_committee_process_worked_plain']}\n\n"
    "ROLE-BY-ROLE WALKTHROUGH\n"
    f"{obj['role_by_role_walkthrough_plain']}\n\n"
    "VOTE AND POLICY\n"
    f"{obj['vote_and_policy_plain']}\n\n"
    "DISSENT AND OBJECTIONS\n"
    f"{obj['dissent_and_objections_plain']}\n\n"
    "GATES AND CONTROLS\n"
    f"{obj['gates_and_controls_plain']}\n\n"
    "OPEN ITEMS / WHAT NEEDS VERIFICATION\n"
    f"{obj['what_is_open_and_needs_verification_plain']}\n\n"
    "WHAT WE DID NOT DO (BOUNDARIES)\n"
    f"{obj['what_we_did_not_do_plain']}\n\n"
    "GLOSSARY\n"
    f"{obj['glossary_plain']}\n"
)

write_text("artifacts/trace_explained_report.txt", narrative)

print("OK — WROTE:")
print(" - artifacts/trace_explained_report.json")
print(" - artifacts/trace_explained_report.txt")
print("If artifacts/trace_explained_report_raw.txt exists, it captured the non-JSON model output for inspection.")

OK — WROTE:
 - artifacts/trace_explained_report.json
 - artifacts/trace_explained_report.txt
If artifacts/trace_explained_report_raw.txt exists, it captured the non-JSON model output for inspection.


In [14]:
# DISPLAY THE TRACE EXPLANATION DIRECTLY IN COLAB (run this after the generator cell)
import json, pathlib
from IPython.display import display, Markdown, JSON

txt_path = pathlib.Path("artifacts/trace_explained_report.txt")
json_path = pathlib.Path("artifacts/trace_explained_report.json")

if not txt_path.exists() and not json_path.exists():
    raise FileNotFoundError("No trace explanation found. Run the trace-explanation generator cell first.")

print("FOUND:")
if txt_path.exists(): print(" -", txt_path)
if json_path.exists(): print(" -", json_path)
print()

# 1) Render the plain-language narrative as Markdown (best for Board-style reading)
if txt_path.exists():
    text = txt_path.read_text(encoding="utf-8")
    display(Markdown("```text\n" + text + "\n```"))

# 2) Also show the structured JSON (expandable)
if json_path.exists():
    obj = json.loads(json_path.read_text(encoding="utf-8"))
    display(Markdown("**Structured JSON (expandable):**"))
    display(JSON(obj))

FOUND:
 - artifacts/trace_explained_report.txt
 - artifacts/trace_explained_report.json



```text
RUN ID: 7492e5e00d94ca0b
TIMESTAMP (UTC): 2026-02-20T20:41:33.027260+00:00
VERIFICATION STATUS: Not verified

EXECUTIVE SUMMARY
The trace explanation could not be generated as valid structured JSON from the model output. This is treated as a governance failure and requires HUMAN_REVIEW. A minimal explanation is provided to preserve process continuity.

WHAT INPUTS WERE USED
Inputs were taken only from the notebook artifacts (run_manifest, reasoning_trace, risk_log, final_report). No external market data, tickers, or sources were used.

HOW THE COMMITTEE PROCESS WORKED
The pipeline requested one structured memo per role (Portfolio, Risk, Compliance/Suitability, Macro). A supervisor then synthesized a vote tally, dissent log, and escalation recommendation. Gates enforced role completeness, dissent preservation, and compliance policy rules.

ROLE-BY-ROLE WALKTHROUGH
Role-by-role details could not be reliably reconstructed into the required schema from the model output. Use artifacts/reasoning_trace.json directly for the canonical per-role memos.

VOTE AND POLICY
The final decision must follow the policy gates recorded in the trace. If Compliance/Suitability stance is REJECT, policy requires escalation (HUMAN_REVIEW or REJECT).

DISSENT AND OBJECTIONS
Dissent must be preserved when roles disagree. Consult artifacts/reasoning_trace.json for the dissent_log field.

GATES AND CONTROLS
Gates are recorded in the trace: role completeness, dissent preservation, and policy enforcement. If any gate fails or if high-severity risks exist, the run should escalate to HUMAN_REVIEW.

OPEN ITEMS / WHAT NEEDS VERIFICATION
Open items and questions to verify are listed in artifacts/final_report.json and must be assigned owners. Verification status remains Not verified until humans validate assumptions and conditions.

WHAT WE DID NOT DO (BOUNDARIES)
This pipeline does not validate real market data, does not forecast returns, and does not provide fiduciary advice. It produces a governed reasoning record and escalates uncertainty for human decision-makers.

GLOSSARY
Trace: the structured record of role memos, votes, dissent, gates, and decision.
Gate: a control check that can force escalation.
Dissent: minority objections preserved verbatim.
Not verified: output must be validated by humans before use.

```

**Structured JSON (expandable):**

<IPython.core.display.JSON object>

In [15]:
# OPTIONAL: SHOW ONLY THE MOST IMPORTANT SECTIONS (clean board-ready view)
import json, pathlib
from IPython.display import display, Markdown

json_path = pathlib.Path("artifacts/trace_explained_report.json")
if not json_path.exists():
    raise FileNotFoundError("Missing artifacts/trace_explained_report.json. Run the generator first.")

obj = json.loads(json_path.read_text(encoding="utf-8"))

sections = [
    ("Executive Summary", "executive_summary_plain"),
    ("What Inputs Were Used", "what_inputs_were_used_plain"),
    ("How the Committee Process Worked", "how_the_committee_process_worked_plain"),
    ("Role-by-Role Walkthrough", "role_by_role_walkthrough_plain"),
    ("Vote and Policy", "vote_and_policy_plain"),
    ("Dissent and Objections", "dissent_and_objections_plain"),
    ("Gates and Controls", "gates_and_controls_plain"),
    ("Open Items / Needs Verification", "what_is_open_and_needs_verification_plain"),
    ("What We Did Not Do", "what_we_did_not_do_plain"),
    ("Glossary", "glossary_plain"),
]

md = []
md.append(f"**RUN ID:** {obj.get('run_id','')}")
md.append(f"**TIMESTAMP (UTC):** {obj.get('timestamp_utc','')}")
md.append(f"**VERIFICATION STATUS:** {obj.get('verification_status','Not verified')}")
md.append("")
for title, key in sections:
    md.append(f"**{title}**")
    md.append(obj.get(key, "").strip() or "(missing)")
    md.append("")

display(Markdown("\n\n".join(md)))

**RUN ID:** 7492e5e00d94ca0b

**TIMESTAMP (UTC):** 2026-02-20T20:41:33.027260+00:00

**VERIFICATION STATUS:** Not verified



**Executive Summary**

The trace explanation could not be generated as valid structured JSON from the model output. This is treated as a governance failure and requires HUMAN_REVIEW. A minimal explanation is provided to preserve process continuity.



**What Inputs Were Used**

Inputs were taken only from the notebook artifacts (run_manifest, reasoning_trace, risk_log, final_report). No external market data, tickers, or sources were used.



**How the Committee Process Worked**

The pipeline requested one structured memo per role (Portfolio, Risk, Compliance/Suitability, Macro). A supervisor then synthesized a vote tally, dissent log, and escalation recommendation. Gates enforced role completeness, dissent preservation, and compliance policy rules.



**Role-by-Role Walkthrough**

Role-by-role details could not be reliably reconstructed into the required schema from the model output. Use artifacts/reasoning_trace.json directly for the canonical per-role memos.



**Vote and Policy**

The final decision must follow the policy gates recorded in the trace. If Compliance/Suitability stance is REJECT, policy requires escalation (HUMAN_REVIEW or REJECT).



**Dissent and Objections**

Dissent must be preserved when roles disagree. Consult artifacts/reasoning_trace.json for the dissent_log field.



**Gates and Controls**

Gates are recorded in the trace: role completeness, dissent preservation, and policy enforcement. If any gate fails or if high-severity risks exist, the run should escalate to HUMAN_REVIEW.



**Open Items / Needs Verification**

Open items and questions to verify are listed in artifacts/final_report.json and must be assigned owners. Verification status remains Not verified until humans validate assumptions and conditions.



**What We Did Not Do**

This pipeline does not validate real market data, does not forecast returns, and does not provide fiduciary advice. It produces a governed reasoning record and escalates uncertainty for human decision-makers.



**Glossary**

Trace: the structured record of role memos, votes, dissent, gates, and decision.
Gate: a control check that can force escalation.
Dissent: minority objections preserved verbatim.
Not verified: output must be validated by humans before use.



##11.CONCLUSION

**CONCLUSION (Board-Facing) — What This Notebook Achieved, What It Cannot Do Yet, and the Bridge Forward**

This notebook makes a specific positive contribution: it turns “AI-assisted analysis” from an opaque text generation activity into a governed committee workflow that produces reviewable evidence. The core achievement is not any single recommendation; it is the architecture that allows a Board, a Risk Committee, and Compliance to inspect the reasoning process as a structured object. We have taken something that is normally informal—discussion, subjective weighting, and implicit assumptions—and converted it into a pipeline with explicit boundaries, explicit roles, explicit decision policies, explicit control gates, and an auditable artifact bundle created on every run.

**Positive contribution: mechanism over mystique**
The notebook demonstrates a key institutional posture: we do not “trust the model”; we trust a controlled process. Four independent roles each generate a structured memo with a stance (APPROVE/REJECT/REVIEW), arguments, objections, required conditions, and risk flags. A supervisor synthesizes these into a committee record that includes a vote tally, a dissent log, and an escalation statement. Most importantly, dissent is not treated as noise—it is treated as evidence. When disagreement exists, the system requires that minority objections are preserved in a specific field, so the committee cannot accidentally “average away” the most important warnings.

**Governance is embedded, not appended**
This notebook enforces governance through gates. Role completeness prevents partial committees. Dissent preservation prevents misleading consensus narratives. Policy enforcement—especially a compliance veto—prevents the system from producing a “green light” outcome when suitability or compliance concerns are unresolved. These are not academic controls; they reflect how real institutions must behave when reputational, regulatory, and fiduciary obligations are in scope.

**Audit artifacts: a minimum deliverable standard**
A major benefit is reproducibility. Every run produces a manifest, prompt hashes (with redaction), a reasoning trace, a risk log, and a final report that separates facts from assumptions and lists open questions. This bundle can be archived, shared, and re-reviewed. If the organization later needs to answer “what did we know and why did we decide,” we have evidence rather than memory.

**What the notebook did (and why it matters)**
- It implemented a multi-role committee reasoning pattern with strict schemas.
- It enforced boundaries so the model cannot quietly introduce external “facts.”
- It created a decision outcome linked to explicit policy rules.
- It produced a board-facing report that is structurally safe: facts vs assumptions vs open items are separated, and verification is explicitly “Not verified.”

This matters because it upgrades our operating model. Instead of analysts producing unstructured memos that vary by author and cannot be audited, we have a standardized committee record format. That standardization is how institutions scale decision quality and defensibility.

**What the notebook cannot do yet (limitations by design)**
This notebook is intentionally not a full investment system. It cannot:
- Validate or ingest real market data with provenance controls.
- Perform portfolio optimization, scenario simulations, stress testing, or execution cost modeling.
- Confirm suitability or compliance outcomes beyond the bounded synthetic inputs.
- Replace human fiduciary judgment or committee sign-off.
- Guarantee that the model’s qualitative reasoning is correct—only that it is captured, constrained, and reviewable.

In addition, while schemas reduce failure risk, they do not eliminate it. A model can still produce plausible but weak arguments; the governance design ensures those arguments are visible, not silently adopted.

**Areas for improvement (the practical roadmap)**
1) Stronger grounding: integrate controlled data connectors with provenance (what source, what timestamp, what transformations).  
2) Quantitative extensions: add deterministic stress tests, drawdown simulations, and concentration analytics that are computed (not narrated).  
3) Human-in-the-loop workflow: add explicit sign-off capture, action owners for open items, and committee minutes generation.  
4) Control escalation: add rule-based triggers that automatically route to Compliance, Legal, or Risk oversight when certain flags appear.  
5) Evaluation harness: add regression tests so the committee record quality can be monitored over time (schema pass rates, assumption leakage, dissent capture rates).

**Bridge to future chapters**
This notebook establishes the committee pattern as an institutional primitive: multiple views, explicit dissent, explicit policy veto, auditable record. The natural next step is to connect this pattern to more advanced reasoning architectures that address the realities of live operations:
- Event-driven pipelines (so the system updates when regimes shift or constraints change).
- Multi-scenario trees and stress-test branches (so the committee sees decision robustness).
- Iterative loops with convergence rules (so the system can refine open items without spinning).
- Trainable evaluation harnesses (so we can measure improvement safely without “mythology”).

In short, this chapter proves we can build AI-assisted committee reasoning as a governed, reviewable mechanism. The next chapters will expand the mechanism’s coverage—more data, more scenarios, more automation—while keeping the same governance spine: bounded inputs, explicit roles, explicit gates, explicit artifacts, and human accountability at the point of decision.